In [1]:
import grewpy
from grewpy import Corpus, CorpusDraft, Request
from collections import Counter
import sys
sys.path.insert(1, "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/tod")

import tod.corpus

treebank_path = "/Users/madalina/Documents/PHD/code/data/SUD_French-Rhapsodie-master/prosody_pauses"
# treebank_path = "rhapsody_test.conllu"
# treebank_path = "/Users/madalina/Downloads/SUD_French-GSD"
# treebank_path = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/input/Universal_Dependencies/ud-treebanks-v2.15/UD_Chinese-GSD"
# treebank_path = "/Users/madalina/Documents/PHD/code/data/test/UD_Beja-Autogramm"
# treebank_path = "/Users/madalina/Downloads/bUD_English-GUM"
# treebank_path = "/Users/madalina/Documents/PHD/code/data/test/UD_Apurina-UFPA"
# treebank_path = "/Users/madalina/Documents/PHD/code/data/test/UD_Romanian-RRT"
grew_pattern = "pattern{X[upos<>PUNCT]}"
patterns_text_file = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/scripts/3. probability_matrix/patterns_all_nodes_prosody.txt"
analysed_category = "all_nodes"

# corpus = tod.corpus.Corpus(
#     treebank_path=treebank_path,
#     grew_pattern=grew_pattern,
#     patterns_text_file=patterns_text_file,
#     use_sud=True,
#     matrix_type="coverage",
#     # excluded_feature_patterns=[r"CxnElt=", r"Cxn=", r"own", r"Gender"]
#         # excluded_feature_patterns=[r"CxnElt=", r"Cxn=", r"XML=", r"PDTB=", r"SplitAnte=", r"MSeg=", r"Entity=", r"Discourse=", r"Bridge=", r"own", r"Degree=Pos"]
# excluded_feature_patterns=[r"own"],
# included_feature_patterns=[r"upos=", 
#                                 r"position=", 
#                                 r"rel_shallow=",
#                                 r"Abbr=", 
#                                 r"Aspect=",  
#                                 r"Animacy=",  
#                                 r"Case=",  
#                                 r"Clusivity=",  
#                                 r"Definite=",  
#                                 r"Deixis=", 
#                                 r"DeixisRef",  
#                                 r"Evident=",  
#                                 r"Negation=",
#                                 r"Number=",  
#                                 r"Gender=",  
#                                 r"Degree=",  
#                                 r"ExtPos=", 
#                                 r"Foreign=", 
#                                 r"Mood",  
#                                 r"NounClass=",  
#                                 r"NumType=",  
#                                 r"Person=",  
#                                 r"Polarity=", 
#                                 r"Polite=",  
#                                 r"Poss=",  
#                                 r"PronType=",  
#                                 r"Reflex=",  
#                                 r"Tense=",  
#                                 r"Typo=",     
#                                 r"VerbForm=",  
#                                 r"Voice=",   ]
#     # )

connected to port: 55464


In [2]:
import numpy as np

# ── 1. FEATURE CLASSIFICATION ────────────────────────────────────────────────

# Drop entirely: redundant, unreliable, or too fine-grained
FEATURES_TO_DROP = {
    'MeanF0',           # use SemitonesFromUtteranceMean instead (speaker-normalised)
    'MeanF0Step',       # threshold artefact, not a prosodic observation
    'AvgAmplitude',     # recording-condition-dependent
    'MaxAmplitude',     # same
    'SylForm',          # phonetic transcription, too fine-grained
    'AlignBegin',       # timing offset, not linguistic
    'AlignEnd',         # timing offset, not linguistic
    'pauseID',          # identifier, not a feature
    # Before*/Next* equivalents of dropped features:
    'BeforeMeanF0', 'BeforeMeanF0Step', 'BeforeAvgAmplitude',
    'NextMeanF0',   'NextMeanF0Step',   'NextAvgAmplitude',
    'Semiton', 'Overlap', 'Loc', 'Glo', 'Backchannel'
}

# Bin into 3 quantile-based categories: these are continuous but informative
# Note: Duration appears in TWO units in your data:
#   - syllable subnodes: integer ms  (e.g. 289, 167)
#   - pause nodes:       float seconds (e.g. 0.2695, 0.3328)
# We normalise everything to ms before binning.
FEATURES_TO_BIN = {
    'SemitonesFromUtteranceMean',
    'Duration',   # needs unit normalisation — see _normalise_duration() below
    # Before*/Next* equivalents:
    'BeforeSemitonesFromUtterance',
    'NextSemitonesFromUtterance',
    'BeforeDuration',   # if present
    'NextDuration',     # if present
    'BeforeSemiton',
    'NextSemiton',
    'NextSemitonsFromUtterance'
    
    
}

# Everything else (Glo, Loc, Slope*, AvgHeight*, PitchRange*, Fused,
# SilentPauseType, ExtraNucleus, HesitationPauseType, NucleusType, …)
# is already categorical → keep as-is, no changes needed.

In [3]:
def _normalise_duration(value_str: str) -> float:
    """Convert Duration string to milliseconds regardless of source node type.
    Syllable subnodes store ms (e.g. '289').
    Pause nodes store seconds (e.g. '0.2695').
    Heuristic: values < 10 are in seconds.
    Zero or invalid → return None (missing / sentinel value).
    """
    try:
        v = float(value_str)
    except (ValueError, TypeError):
        return None
    if v == 0.0:
        return None   # sentinel for missing data (e.g. overlap nodes)
    return v * 1000 if v < 10 else v   # seconds → ms if < 10


In [4]:
def _is_sentinel(value_str: str) -> bool:
    """Catch the '0.0' sentinel values that appear on overlap/missing pause nodes.
    example from test file: node 32's pause has all-zero Before*/Next* values because the audio overlaps with another sentence. we want to catch these and drop them silently rather than binning them as F0_low.
    """
    try:
        return float(value_str) == 0.0
    except (ValueError, TypeError):
        return False

In [5]:
def _feature_name(key: tuple) -> str:
    """Extract the bare feature name from a key like ('node','X','child','Duration')."""
    return key[-1]

In [6]:
def collect_continuous_values(all_data_items):
    """
    all_data_items: iterable of (lex_unit, match, features) triples
    where features is the dict returned by grex.data.extract_features().
    
    Returns a dict: {feature_name: [float, float, ...]} with all observed values.
    """
    collector = {f: [] for f in FEATURES_TO_BIN}

    for _, _, features in all_data_items:
        for key, value in features.items():
            fname = _feature_name(key)
            if fname not in FEATURES_TO_BIN:
                continue
            # value can be a plain string (own/parent/prev/next)
            # or a set of strings (child, which can have multiple syllables)
            values = value if isinstance(value, set) else {value}
            for v in values:
                if fname == 'Duration':
                    norm = _normalise_duration(v)
                    if norm is not None:
                        collector['Duration'].append(norm)
                else:
                    if not _is_sentinel(v):
                        try:
                            collector[fname].append(float(v))
                        except (ValueError, TypeError):
                            pass
    return collector

In [7]:
def compute_thresholds(collector: dict, n_bins: int = 3) -> dict:
    """
    Returns a dict: {feature_name: [boundary1, boundary2, ...]}
    with (n_bins - 1) boundaries derived from corpus quantiles.
    """
    thresholds = {}
    for fname, values in collector.items():
        if not values:
            continue
        arr = np.array(values)
        quantiles = np.linspace(0, 1, n_bins + 1)[1:-1]   # interior boundaries only
        thresholds[fname] = np.quantile(arr, quantiles).tolist()
    return thresholds

In [8]:
BIN_LABELS = {
    'SemitonesFromUtteranceMean':    ['F0_low',    'F0_mid',    'F0_high'],
    'Duration':                      ['Dur_short', 'Dur_mid',   'Dur_long'],
    'BeforeSemitonesFromUtterance':  ['BeforeF0_low',  'BeforeF0_mid',  'BeforeF0_high'],
    'NextSemitonesFromUtterance':    ['NextF0_low',    'NextF0_mid',    'NextF0_high'],
}

In [9]:
def _bin_value(fname: str, value_str: str, thresholds: dict) -> str | None:
    """
    Convert a continuous string value to a bin label.
    Returns None if the value should be skipped (sentinel, missing, out-of-range).
    """
    if fname not in thresholds:
        return None
    if _is_sentinel(value_str):
        return None

    if fname == 'Duration':
        v = _normalise_duration(value_str)
    else:
        try:
            v = float(value_str)
        except (ValueError, TypeError):
            return None

    if v is None:
        return None

    bounds = thresholds[fname]
    labels = BIN_LABELS.get(fname, [f'{fname}_Q1', f'{fname}_Q2', f'{fname}_Q3'])
    # find which bin: below first boundary → label[0], etc.
    for i, b in enumerate(bounds):
        if v <= b:
            return labels[i]
    return labels[-1]

In [10]:
def format_features(features: dict, thresholds: dict) -> list[str]:
    """
    Replace the original formatted_features list comprehension.
    
    features:   the dict from grex.data.extract_features()
    thresholds: computed by compute_thresholds() over the full corpus
    
    Returns a list of 'key1:key2:...:keyN=value' strings,
    with continuous features binned and dropped features excluded.
    """
    result = []

    for key, value in features.items():
        fname = _feature_name(key)
        prefix = ':'.join(key)

        # ── DROP ──────────────────────────────────────────────────────────────
        if fname in FEATURES_TO_DROP:
            continue

        # ── BIN (continuous → categorical label) ─────────────────────────────
        if fname in FEATURES_TO_BIN:
            values = value if isinstance(value, set) else {value}
            for v in values:
                label = _bin_value(fname, v, thresholds)
                if label is not None:
                    result.append(f"{prefix}={label}")
            continue

        # ── KEEP AS-IS (already categorical) ─────────────────────────────────
        if isinstance(value, set):
            for v in value:
                result.append(f"{prefix}={v}")
        else:
            result.append(f"{prefix}={value}")

    return result

In [21]:
def _filter_features(features, excluded_patterns=None, included_patterns=None):
    import re
    filtered_features = []
    for feature in features:
        # Check exclusion patterns
        should_exclude = False
        if excluded_patterns:
            for pattern in excluded_patterns:
                if re.search(pattern, feature):  # Use 'pattern', not 'excluded_patterns'
                    should_exclude = True
                    break
        if should_exclude:
            continue
        
        # Check inclusion patterns (only if not excluded)
        should_include = True
        if included_patterns:
            should_include = False
            for pattern in included_patterns:
                if re.search(pattern, feature):  # Use 'pattern', not 'included_patterns'
                    should_include = True
                    break
        if should_include:
            filtered_features.append(feature)
    return filtered_features

In [22]:
import yaml
sys.path.insert(1, "/Users/madalina/Documents/M2TAL/stage/grex/grex2")
import pyximport
import pandas as pd
pyximport.install()
import grex.data
import grex.utils
import grex.features
import grex.data_prosody

grewpy.set_config("sud")
corpus = grewpy.Corpus(treebank_path)
draft = grewpy.CorpusDraft(corpus)
all_matches = corpus.search(
            grewpy.Request(grew_pattern)
            .without("X[InIdiom=Yes]")
            .without("X[Idiom=Yes]")
            .without("X[InTitle=Yes]")
            .without("X[Title=Yes]")
            .without("X[Scrap=Yes]")
            .without("X[Foreign]")
            .without("X[Lang]")
            .without("X-[fixed]->Y")
            .without("Y-[flat:name]->X")
            .without("Y-[goeswith]->X"),
            clustering_parameter=["X.lemma"],
        )
sent_id_to_sentence = {
            draft[i].meta["sent_id"]: draft[i].features for i in range(len(draft)) # type: ignore
        }

match_upos = {}
        # iterating through each lemma and its values like this: '€': [{'sent_id': 'fr-ud-train_11309', 'matching': {'nodes': {'X': '7'}, 'edges': {}}}]
for key, value in all_matches.items():
    # iterating through each match for the lemma so for example {'sent_id': 'fr-ud-train_11309', 'matching': {'nodes': {'X': '7'}, 'edges': {}}}
    for m in value:
        match_sent_id = m["sent_id"]
        match_node_index = str(m["matching"]["nodes"]["X"])
        if match_sent_id in sent_id_to_sentence:
            current_sentence_features = sent_id_to_sentence[match_sent_id]
            if match_node_index in current_sentence_features.keys():
                current_token_features = current_sentence_features[
                    match_node_index
                ]
                # if the token has a feature called ExtPos, we use that as the key, otherwise we use the upos feature
                # ExtPos is used for example in 10% -> % has upos SYM but ExtPos Noun , so we want to use noun
                # and what we're doing is creating a dictionary with keys (lemma, pos) and values as a list of matches
                # so for example ('reason', 'NOUN') -> [match1, match2, match3] where a match looks like this: {'sent_id': 'fr-ud-train_11309', 'matching': {'nodes': {'X': '7'}, 'edges': {}}}
                
                if "ExtPos" in current_token_features:
                    match_upos.setdefault(
                        (key, current_token_features["ExtPos"]), []
                    ).append(m)
                else:
                    try:
                        match_upos.setdefault(
                            (key, current_token_features["upos"]), []
                        ).append(m)
                    except KeyError:
                        print(f"WARNING: Token with lemma '{key}' in sentence '{match_sent_id}' does not have a 'upos' feature.")

with open(patterns_text_file) as in_stream:
            config = yaml.load(in_stream, Loader=yaml.Loader)

templates = grex.utils.FeaturePredicate.from_config(config["templates"])
feature_predicate = grex.utils.FeaturePredicate.from_config(
    config["features"], templates=templates
)

all_data_items = []
for lex_unit, mts in match_upos.items():
    for match in mts:
        features = grex.data_prosody.extract_features(draft, match, feature_predicate)
        all_data_items.append((lex_unit, match, features))

collector  = collect_continuous_values(all_data_items)
thresholds = compute_thresholds(collector, n_bins=3)

print("Computed thresholds:")
for fname, bounds in thresholds.items():
    labels = BIN_LABELS.get(fname, ['Q1','Q2','Q3'])
    print(f"  {fname}: <{bounds[0]:.3f} → {labels[0]} | "
          f"<{bounds[1]:.3f} → {labels[1]} | ≥{bounds[1]:.3f} → {labels[2]}")

# --- SECOND PASS: format with binning applied ---
data = {k: list() for k in match_upos}

for lex_unit, match, features in all_data_items:
    formatted = format_features(features, thresholds)
    # if excluded_feature_patterns or included_feature_patterns:
    formatted = _filter_features(
        formatted,
        excluded_patterns=['own'],
        # included_patterns=included_feature_patterns
    )
    data[lex_unit].append(formatted)

unique_lemma = sorted(set([k for k in data]))
unique_features = sorted(set(
    feat
    for _, match_upos in data.items()
    for m in match_upos
    for feat in m
))

idx2feature = {i: feat for i, feat in enumerate(unique_features)}
feature2idx = {feat: i for i, feat in idx2feature.items()}
idx2lexunit = {i: feat for i, feat in enumerate(unique_lemma)}
lexunit2idx = {feat: i for i, feat in idx2lexunit.items()}

Computed thresholds:
  Duration: <130.000 → Dur_short | <190.000 → Dur_mid | ≥190.000 → Dur_long
  SemitonesFromUtteranceMean: <-1.266 → F0_low | <0.529 → F0_mid | ≥0.529 → F0_high
